# Deliverable 4 — Gaussian HMM market-regime filtering (final submission)

**Research question:** Does Gaussian Hidden Markov Model (HMM) regime-based filtering improve the risk-adjusted performance of a 20/50-day SMA crossover strategy and a 20-day breakout strategy relative to unfiltered trading?

**Final experiment:** S&P 500 (`^GSPC`), 2000–2024 raw series. The 2000–2005 development period is excluded from final model training/testing **rows**. Rolling HMM fitting uses **31 observations** (the inclusive days 91–121 worked example in Deliverable 3; its prose also says “30-day,” an acknowledged inconsistency), with training-only scaling, feature correlation filtering at `|r| > 0.80`, and BIC selection among 2-, 3- and 4-state Gaussian HMMs. Historical favourable states use mean next-open-to-next-open return on the training observations. Signals are generated at close `t`; positions are entered at open `t+1` and evaluated through open `t+2`. Cost = **5 bps fees + 2 bps slippage per position change**.

**Final out-of-sample predictions:** 4,749 dates, 2006-02-16 through 2024-12-30. **Backtested periods:** 4,747, with execution dates 2006-02-17 through 2024-12-27. The underlying S&P 500 cash index is not directly tradable; the backtest is a hypothetical index-price simulation.

### Instructions

1. Keep this notebook in the **same project directory** as `hmm_dataset.csv` and the `final_report_outputs/` folder.
2. In VS Code choose **Run All**. The completed checkpoint is loaded; full HMM retraining is **disabled by default**, to avoid repeating thousands of fits.
3. If reproducing the experiment from scratch, change `RUN_HMM_TRAINING` to `True` in the optional fitting cell. It creates/resumes checkpoints of the *same* method; run that cell again until all windows are complete, then run the evaluation cells.
4. Submission evidence should include `final_report_outputs/methodology_aligned_predictions_strict.csv`, audit files, `hmm_dataset.csv`, the final written report, this notebook and the dependency/version manifest. Do not mislabel the earlier `hmm_integrated_results.csv` experiment as the final one.

**Integrity:** The first version of this notebook was rebuilt from the uploaded source notebook and the completed methodology-aligned run. It uses the user's saved checkpoint for the published final numbers, not any guessed output. **Do not edit the model or the research question solely to change returns.**

## 1. Load and validate raw S&P 500 data

The raw export has two non-data header rows. Prices are parsed, dates sorted and duplicate timestamps removed. Raw data itself is preserved.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Load the dataset
data = pd.read_csv(
    "hmm_dataset.csv",
    skiprows=2,
    header=None,
    names=["Date", "Close", "High", "Low", "Open", "Volume"]
)

# Convert columns to the correct data types
data["Date"] = pd.to_datetime(
    data["Date"], format="%d/%m/%Y", errors="coerce"
)

for column in ["Close", "High", "Low", "Open", "Volume"]:
    data[column] = pd.to_numeric(data[column], errors="coerce")

# Clean and organise the data
data = (
    data.dropna(subset=["Date"])
        .sort_values("Date")
        .drop_duplicates(subset=["Date"])
        .set_index("Date")
)

# Display the results
print("Dataset shape:", data.shape)
print("Start date:", data.index.min())
print("End date:", data.index.max())

print("\nMissing values:")
print(data.isnull().sum())

print("\nFirst five rows:")
display(data.head())


## 2. Engineer causal daily features

Log returns, 20-day realised volatility, 20/50 SMAs and ratio, RSI(14), and ROC(14) are calculated using current and past prices. The six engineered indicators are created first; **five** candidate HMM features are then retained as proposed by the supervisor. Pairwise correlation filtering occurs **inside each individual training window**, not at this full-sample stage.

In [ ]:
# Calculate daily log returns
data["Log_Returns"] = np.log(
    data["Close"] / data["Close"].shift(1)
)

# Keep the Returns column for the existing EDA cells,
# using log returns as specified in Deliverable 3
data["Returns"] = data["Log_Returns"]

# 20-day realised volatility of log returns
data["Volatility"] = data["Log_Returns"].rolling(
    window=20
).std()

# 20-day and 50-day simple moving averages
data["SMA_20"] = data["Close"].rolling(window=20).mean()
data["SMA_50"] = data["Close"].rolling(window=50).mean()

# Ratio of the two moving averages
data["SMA_Ratio"] = data["SMA_20"] / data["SMA_50"]

# 14-day Rate of Change (ROC)
data["ROC_14"] = (
    (data["Close"] / data["Close"].shift(14)) - 1
)

# 14-day Relative Strength Index (RSI)
price_change = data["Close"].diff()

gains = price_change.clip(lower=0)
losses = -price_change.clip(upper=0)

average_gain = gains.ewm(
    alpha=1/14, min_periods=14, adjust=False
).mean()

average_loss = losses.ewm(
    alpha=1/14, min_periods=14, adjust=False
).mean()

relative_strength = average_gain / average_loss

data["RSI_14"] = 100 - (
    100 / (1 + relative_strength)
)

# Prepare the complete engineered dataset
candidate_features = [
    "Volatility",
    "SMA_20",
    "SMA_50",
    "SMA_Ratio",
    "RSI_14",
    "ROC_14"
]

features_data = data.dropna(
    subset=candidate_features + ["Log_Returns"]
).copy()

print("Dataset shape after feature engineering:", features_data.shape)
print("\nEngineered features:")
display(features_data[candidate_features].head())

print("\nFeature summary:")
display(features_data[candidate_features].describe())


In [ ]:
# Five candidate HMM inputs; SMA_20 is retained for trading rules.
CANDIDATE_HMM_FEATURES = [
    "Volatility", "SMA_50", "SMA_Ratio", "RSI_14", "ROC_14"
]
model_data = features_data[CANDIDATE_HMM_FEATURES].copy()
assert not model_data.isna().any().any()
assert np.isfinite(model_data.to_numpy()).all()
assert len(data) == 6288, "Raw dataset length differs from verified input."
print("Model feature observations:", len(model_data))
print("Model date range:", model_data.index.min(), "through", model_data.index.max())


## 3. Build the two fixed technical strategies

SMA long signal when SMA(20) exceeds SMA(50); the 20-day breakout enters/exits using previous 20 closing-price extrema. The breakout holds its previous long/flat state between triggers. Both strategies are defined independently of the HMM.

In [ ]:
# Create the trading strategy signals
strategy_data = features_data[
    ["Open", "Close", "SMA_20", "SMA_50"]
].copy()

# Strategy 1: SMA crossover
# Long when the 20-day SMA is above the 50-day SMA.
# Otherwise remain out of the market.
strategy_data["SMA_Signal"] = (
    strategy_data["SMA_20"] > strategy_data["SMA_50"]
).astype(int)

# Strategy 2: 20-day breakout
# Thresholds use previous closing prices only.
strategy_data["Breakout_High_20"] = (
    data["Close"].shift(1).rolling(20).max()
    .reindex(strategy_data.index)
)

strategy_data["Breakout_Low_20"] = (
    data["Close"].shift(1).rolling(20).min()
    .reindex(strategy_data.index)
)

# Maintain the existing position until an exit occurs.
breakout_positions = []
current_position = 0

for _, row in strategy_data.iterrows():
    if row["Close"] > row["Breakout_High_20"]:
        current_position = 1
    elif row["Close"] < row["Breakout_Low_20"]:
        current_position = 0

    breakout_positions.append(current_position)

strategy_data["Breakout_Signal"] = breakout_positions

# Verify the generated trading signals.
print("Trading strategy dataset:", strategy_data.shape)
print("Date range:", strategy_data.index.min(),
      "to", strategy_data.index.max())

print("\nSMA signal distribution:")
print(strategy_data["SMA_Signal"].value_counts())

print("\nBreakout signal distribution:")
print(strategy_data["Breakout_Signal"].value_counts())

display(strategy_data.tail(10))


## 4. Fixed experimental settings and correlation filtering

These parameters and the feature-priority rule are specified **before examining the revised strategy returns**. The priority order resolves within-window correlations deterministically; no market outcome or future observation determines the retained features.

In [ ]:
INITIAL_WINDOW = 31
WARMUP_DAYS = 90
HMM_STATE_COUNTS = (2, 3, 4)
CORRELATION_LIMIT = 0.80
FEATURE_PRIORITY = ["Volatility", "SMA_Ratio", "RSI_14", "SMA_50", "ROC_14"]
TRANSACTION_COST = 0.0005
SLIPPAGE = 0.0002
TOTAL_COST = TRANSACTION_COST + SLIPPAGE
OUTPUT_DIR = Path("final_report_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PREDICTION_FILE = OUTPUT_DIR / "methodology_aligned_predictions_strict.csv"


def select_training_features(train_window):
    # Correlations use this training window only; no test information.
    corr = train_window[FEATURE_PRIORITY].corr().abs()
    retained = []
    for feature in FEATURE_PRIORITY:
        if all(pd.notna(corr.loc[feature, existing])
               and corr.loc[feature, existing] <= CORRELATION_LIMIT
               for existing in retained):
            retained.append(feature)
    if not retained:
        raise RuntimeError("No features remain after training-only filtering")
    return retained

print("Training window:", INITIAL_WINDOW, "rows")
print("State candidates:", HMM_STATE_COUNTS)
print("Cost per position change:", TOTAL_COST)


## 5. Fit Gaussian HMMs and identify favourable historical states

For each rolling window, StandardScaler is trained on those 31 observations only. HMMs are fitted with 20 random seeds per candidate state count and selected by lowest BIC, with numerical fallback to transition prior 1.01 if all default seeds fail. Historical Viterbi states and the two fixed strategies identify favourable states using **training-period returns only**.

In [ ]:
import warnings
from sklearn.preprocessing import StandardScaler
from hmmlearn.hmm import GaussianHMM

def fit_select_hmm(train_window, seeds=range(20)):
    """
    Fit 2-, 3-, and 4-state Gaussian HMMs.

    Uses the original settings first.
    If every seed fails for a candidate, retry that
    candidate with a small transition prior (1.01).

    Select the best valid candidate using BIC.
    """

    X = train_window.to_numpy(dtype=float)

    if not np.isfinite(X).all():
        raise ValueError("Training data contains invalid values.")

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    n_observations, n_features = X_scaled.shape

    candidate_results = []
    candidate_models = {}

    fallback_candidates = []
    failed_candidates = []

    for n_states in [2, 3, 4]:

        valid_fits = []
        fitting_method = "Default"

        # First use original settings; retry with a
        # transition prior only if every seed fails.
        for transition_prior in [1.0, 1.01]:

            valid_fits = []

            for seed in seeds:

                model = GaussianHMM(
                    n_components=n_states,
                    covariance_type="diag",
                    n_iter=1000,
                    tol=1e-3,
                    min_covar=1e-3,
                    transmat_prior=transition_prior,
                    random_state=seed
                )

                try:
                    with warnings.catch_warnings(record=True) as caught:
                        warnings.simplefilter("always")
                        model.fit(X_scaled)

                    log_likelihood = model.score(X_scaled)
                    converged = model.monitor_.converged

                except (
                    ValueError,
                    FloatingPointError,
                    np.linalg.LinAlgError
                ):
                    continue

                if np.isfinite(log_likelihood):

                    valid_fits.append({
                        "model": model,
                        "seed": seed,
                        "log_likelihood": log_likelihood,
                        "converged": converged,
                        "iterations": model.monitor_.iter,
                        "warnings": len(caught)
                    })

            if valid_fits:
                if transition_prior > 1.0:
                    fitting_method = "Transition prior fallback"
                    fallback_candidates.append(n_states)

                break

        if not valid_fits:
            failed_candidates.append(n_states)

            print(
                f"No valid {n_states}-state model for "
                f"{train_window.index[-1].date()}"
            )
            continue

        converged_fits = [
            fit for fit in valid_fits
            if fit["converged"]
        ]

        if converged_fits:
            eligible_fits = converged_fits
            convergence_status = "Converged fit available"
        else:
            eligible_fits = valid_fits
            convergence_status = "Best finite fit used"

        best_fit = max(
            eligible_fits,
            key=lambda fit: fit["log_likelihood"]
        )

        best_log_likelihood = best_fit["log_likelihood"]

        n_parameters = (
            n_states * (n_states - 1)
            + (n_states - 1)
            + 2 * n_states * n_features
        )

        aic = -2 * best_log_likelihood + 2 * n_parameters

        bic = (
            -2 * best_log_likelihood
            + n_parameters * np.log(n_observations)
        )

        candidate_results.append({
            "States": n_states,
            "Seed": best_fit["seed"],
            "Log_Likelihood": best_log_likelihood,
            "Parameters": n_parameters,
            "AIC": aic,
            "BIC": bic,
            "Converged": best_fit["converged"],
            "Iterations": best_fit["iterations"],
            "Warnings": best_fit["warnings"],
            "Fit_Status": convergence_status,
            "Fitting_Method": fitting_method,
            "Valid_Initialisations": len(valid_fits)
        })

        candidate_models[n_states] = best_fit["model"]

    if not candidate_results:
        raise RuntimeError(
            "All HMM candidates failed, including fallback, "
            f"for window ending {train_window.index[-1].date()}."
        )

    results = (
        pd.DataFrame(candidate_results)
        .sort_values("BIC")
        .reset_index(drop=True)
    )

    results.attrs["failed_candidates"] = failed_candidates
    results.attrs["fallback_candidates"] = fallback_candidates

    selected_states = int(results.iloc[0]["States"])
    selected_model = candidate_models[selected_states]

    return selected_model, scaler, selected_states, results

def identify_favourable_states(
    model, scaler, train_window, strategy_data
):
    # Identify states within this training window
    X_train = scaler.transform(
        train_window.to_numpy(dtype=float)
    )

    train_states = model.predict(X_train)

    # Align strategy observations with HMM training dates
    train_strategies = strategy_data.loc[
        train_window.index
    ]

    opens = train_strategies["Open"].to_numpy(dtype=float)

    # Signal at close of day i:
    # enter at open i+1, earn return through open i+2.
    # Both opening prices must be inside the training window.
    forward_returns = opens[2:] / opens[1:-1] - 1

    # The final two observations cannot be evaluated
    # without opening prices outside the training window.
    usable_states = train_states[:-2]

    favourable = {}

    for strategy_name, signal_column in [
        ("SMA", "SMA_Signal"),
        ("Breakout", "Breakout_Signal")
    ]:

        signals = train_strategies[
            signal_column
        ].to_numpy(dtype=int)[:-2]

        favourable_states = []

        for state in range(model.n_components):

            # Historical observations in this state
            # where the strategy generated a long signal
            eligible = (
                (usable_states == state)
                & (signals == 1)
            )

            if eligible.sum() == 0:
                continue

            mean_return = forward_returns[eligible].mean()

            if mean_return > 0:
                favourable_states.append(state)

        favourable[strategy_name] = favourable_states

    return favourable


## 6. Optional: generate or resume the complete walk-forward checkpoint

**Default is to reuse the completed prediction checkpoint**. This section contains the full fit/selection/filtering algorithm so the experiment remains reproducible, but running 4,749 prediction windows from scratch is computationally expensive.

To rebuild from scratch, move/rename the existing prediction CSV (keep a backup), set `RUN_HMM_TRAINING = True`, and rerun this cell repeatedly until the saved rows reach 4,749. Checkpoint dates must be consecutive and start after the first 31 *post-2005* training observations.

In [ ]:
import os
import time
RUN_HMM_TRAINING = False   # Change to True only to deliberately fit/resume HMMs.
BATCH_SIZE = 500
SAVE_EVERY = 25
first_2006_position = model_data.index.searchsorted(pd.Timestamp("2006-01-01"))
first_position = first_2006_position + INITIAL_WINDOW
expected_dates = pd.DatetimeIndex(model_data.index[first_position:])
total_steps = len(expected_dates)


def load_completed_checkpoint():
    if not PREDICTION_FILE.exists():
        return pd.DataFrame()
    results = pd.read_csv(PREDICTION_FILE)
    results["Prediction_Date"] = pd.to_datetime(results["Prediction_Date"], format="mixed")
    actual = pd.DatetimeIndex(results["Prediction_Date"])
    if not actual.equals(expected_dates[:len(actual)]):
        raise RuntimeError("Checkpoint contains unexpected dates or ordering")
    return results


def append_checkpoint(existing, pending):
    if not pending:
        return existing
    updated = pd.concat([existing, pd.DataFrame(pending)], ignore_index=True)
    temp = PREDICTION_FILE.with_suffix(".tmp")
    updated.to_csv(temp, index=False)
    os.replace(temp, PREDICTION_FILE)
    pending.clear()
    return updated


checkpoint = load_completed_checkpoint()
print("Available checkpoint rows:", len(checkpoint), "of", total_steps)

if RUN_HMM_TRAINING:
    start_step = len(checkpoint)
    end_step = min(start_step + BATCH_SIZE, total_steps)
    pending = []
    started = time.time()
    try:
        for step in range(start_step, end_step):
            pos = first_position + step
            full_train = model_data.iloc[pos-INITIAL_WINDOW:pos]
            prediction_date = model_data.index[pos]
            assert len(full_train) == INITIAL_WINDOW
            assert full_train.index.min() >= pd.Timestamp("2006-01-01")
            feature_names = select_training_features(full_train)
            train = full_train[feature_names]
            next_obs = model_data.iloc[[pos]][feature_names]
            model, scaler, n_states, selection = fit_select_hmm(train)
            favourable = identify_favourable_states(model, scaler, train, strategy_data)
            X_train = scaler.transform(train.to_numpy(dtype=float))
            X_next = scaler.transform(next_obs.to_numpy(dtype=float))
            probs = model.predict_proba(np.vstack([X_train, X_next]))[-1]
            state = int(np.argmax(probs))
            sma_signal = int(strategy_data.loc[prediction_date, "SMA_Signal"])
            breakout_signal = int(strategy_data.loc[prediction_date, "Breakout_Signal"])
            sma_good = int(state in favourable["SMA"])
            breakout_good = int(state in favourable["Breakout"])
            selected = selection.loc[selection["States"] == n_states].iloc[0]
            record = {
                "Prediction_Date": prediction_date,
                "Train_Start": full_train.index[0],
                "Train_End": full_train.index[-1],
                "Feature_Count": len(feature_names),
                "Selected_Features": ", ".join(feature_names),
                "Selected_States": n_states,
                "Raw_Predicted_State": state,
                "State_Probability": float(probs[state]),
                "Selected_BIC": float(selected["BIC"]),
                "Converged": bool(selected["Converged"]),
                "Fitting_Method": selected["Fitting_Method"],
                "SMA_Signal": sma_signal,
                "Breakout_Signal": breakout_signal,
                "SMA_Favourable": sma_good,
                "Breakout_Favourable": breakout_good,
                "SMA_Filtered_Signal": sma_signal * sma_good,
                "Breakout_Filtered_Signal": breakout_signal * breakout_good,
            }
            for candidate_state in range(4):
                record[f"Probability_{candidate_state}"] = (
                    float(probs[candidate_state]) if candidate_state < n_states else np.nan
                )
            pending.append(record)
            if len(pending) >= SAVE_EVERY:
                checkpoint = append_checkpoint(checkpoint, pending)
            if (step+1) % SAVE_EVERY == 0 or step == end_step-1:
                print(f"Completed {step+1}/{total_steps} through {prediction_date.date()}")
    finally:
        checkpoint = append_checkpoint(checkpoint, pending)
    print("Batch elapsed minutes:", round((time.time()-started)/60, 2))
    print("Saved:", len(checkpoint), "Remaining:", total_steps-len(checkpoint))
else:
    print("Full HMM retraining disabled; using the saved final checkpoint.")


## 7. Backtest functions (unchanged trading rules)

At each prediction date, signals come from that date's close and enter at the **next session's open**. Equity accrues through the following open. The initial portfolio position is flat. A **7-basis-point charge** is applied to each entry/exit (position change), without inventing closing trades at the dataset boundary.

In [ ]:
def compare_strategy_returns(integrated_results, strategy_data):

    predictions = integrated_results.copy()
    predictions["Prediction_Date"] = pd.to_datetime(
    predictions["Prediction_Date"],
    format="mixed"
)




    predictions = (
        predictions.sort_values("Prediction_Date")
        .drop_duplicates("Prediction_Date")
        .set_index("Prediction_Date")
    )

    # Every prediction must correspond to a trading day.
    positions = strategy_data.index.get_indexer(
        predictions.index
    )

    if (positions < 0).any():
        raise ValueError("Prediction dates missing from strategy_data.")

    # Signal at close t becomes a position at open t+1.
    execution_positions = positions + 1

    # Need open t+2 to measure the completed holding return.
    valid = execution_positions + 1 < len(strategy_data)

    predictions = predictions.iloc[np.flatnonzero(valid)]
    execution_positions = execution_positions[valid]

    open_prices = strategy_data["Open"].to_numpy(dtype=float)

    market_returns = (
        open_prices[execution_positions + 1]
        / open_prices[execution_positions] - 1
    )

    execution_dates = strategy_data.index[execution_positions]

    comparison = pd.DataFrame(
        {"Market_Return": market_returns},
        index=execution_dates
    )

    comparison.index.name = "Execution_Date"

    strategy_pairs = {
        "SMA_Unfiltered": "SMA_Signal",
        "SMA_Filtered": "SMA_Filtered_Signal",
        "Breakout_Unfiltered": "Breakout_Signal",
        "Breakout_Filtered": "Breakout_Filtered_Signal"
    }

    for name, signal_column in strategy_pairs.items():

        # These are positions held from each execution open.
        held_positions = predictions[
            signal_column
        ].to_numpy(dtype=int)

        # Assume a flat position before the evaluation starts.
        previous_positions = np.r_[
            0, held_positions[:-1]
        ]

        position_changes = np.abs(
            held_positions - previous_positions
        )

        net_returns = (
            held_positions * market_returns
            - position_changes * TOTAL_COST
        )

        comparison[f"{name}_Position"] = held_positions
        comparison[f"{name}_Trades"] = position_changes
        comparison[f"{name}_Return"] = net_returns
        comparison[f"{name}_Equity"] = (
            1 + net_returns
        ).cumprod()

    return comparison

def calculate_performance_metrics(comparison, strategy_name):

    returns = comparison[
        f"{strategy_name}_Return"
    ].astype(float)

    positions = comparison[
        f"{strategy_name}_Position"
    ].astype(int)

    changes = comparison[
        f"{strategy_name}_Trades"
    ].astype(int)

    # Cumulative portfolio growth
    equity = (1 + returns).cumprod()

    cumulative_return = (equity.iloc[-1] - 1) * 100

    # Annualised Sharpe ratio
    # Assumes zero risk-free rate.
    volatility = returns.std(ddof=1)

    sharpe = (
        np.sqrt(252) * returns.mean() / volatility
        if volatility > 0 else np.nan
    )

    # Maximum drawdown, including starting equity of 1.
    equity_with_start = pd.concat([
        pd.Series([1.0]),
        equity.reset_index(drop=True)
    ], ignore_index=True)

    running_peak = equity_with_start.cummax()

    drawdown = (
        equity_with_start / running_peak - 1
    )

    max_drawdown = drawdown.min() * 100

    # Completed trade win rate
    # Each trade begins when position changes 0 -> 1
    # and ends when position changes 1 -> 0.
    trade_returns = []
    trade_growth = 1.0
    in_trade = False

    for position, daily_return in zip(
        positions.to_numpy(),
        returns.to_numpy()
    ):

        if position == 1:
            if not in_trade:
                in_trade = True
                trade_growth = 1.0

            trade_growth *= (1 + daily_return)

        elif in_trade:
            # Include exit transaction cost on the
            # first flat-position day.
            trade_growth *= (1 + daily_return)

            trade_returns.append(trade_growth - 1)

            trade_growth = 1.0
            in_trade = False

    completed_trades = len(trade_returns)

    win_rate = (
        np.mean(np.array(trade_returns) > 0) * 100
        if completed_trades > 0 else np.nan
    )

    return {
        "Strategy": strategy_name,
        "Cumulative_Return_%": cumulative_return,
        "Sharpe_Ratio": sharpe,
        "Max_Drawdown_%": max_drawdown,
        "Completed_Trades": completed_trades,
        "Win_Rate_%": win_rate,
        "Position_Changes": int(changes.sum()),
        "Time_In_Market_%": positions.mean() * 100
    }


## 8. Load and verify the methodology-aligned final checkpoint

The earlier exploratory full-period experiment (`hmm_integrated_results.csv`) is **not** used for the final reported results. Validation requires exactly 4,749 consecutive prediction dates and training windows restricted to dates in 2006 or later.

In [ ]:
corrected_predictions = load_completed_checkpoint()
if len(corrected_predictions) != total_steps:
    raise RuntimeError(
        f"Final prediction checkpoint is incomplete: {len(corrected_predictions)}/{total_steps}. "
        "Either restore the completed CSV or finish optional model fitting first."
    )
for col in ("Prediction_Date", "Train_Start", "Train_End"):
    corrected_predictions[col] = pd.to_datetime(corrected_predictions[col], format="mixed")
assert corrected_predictions["Prediction_Date"].is_unique
assert corrected_predictions["Train_Start"].ge(pd.Timestamp("2006-01-01")).all()
assert corrected_predictions["Train_End"].lt(corrected_predictions["Prediction_Date"]).all()
assert corrected_predictions["Selected_States"].between(2, 4).all()
assert corrected_predictions["Feature_Count"].between(1, 5).all()
assert corrected_predictions["Converged"].astype(str).str.lower().eq("true").all()
assert corrected_predictions["State_Probability"].between(0, 1).all()
actual = pd.DatetimeIndex(corrected_predictions["Prediction_Date"])
assert actual.equals(expected_dates)
locs = model_data.index.get_indexer(actual)
assert (locs >= INITIAL_WINDOW).all()
assert np.array_equal(
    corrected_predictions["Train_Start"].to_numpy(),
    model_data.index[locs-INITIAL_WINDOW].to_numpy()
)
assert np.array_equal(
    corrected_predictions["Train_End"].to_numpy(),
    model_data.index[locs-1].to_numpy()
)

final_comparison = compare_strategy_returns(corrected_predictions, strategy_data)
strategy_names = ["SMA_Unfiltered", "SMA_Filtered", "Breakout_Unfiltered", "Breakout_Filtered"]
final_summary = pd.DataFrame([
    calculate_performance_metrics(final_comparison, name) for name in strategy_names
])
assert len(final_comparison) == 4747
assert final_comparison.index.min() == pd.Timestamp("2006-02-17")
assert final_comparison.index.max() == pd.Timestamp("2024-12-27")
expected_returns = {
    "SMA_Unfiltered": 128.992,
    "SMA_Filtered": 21.577,
    "Breakout_Unfiltered": 98.954,
    "Breakout_Filtered": 40.293
}
for _, row in final_summary.iterrows():
    name = row["Strategy"]
    assert abs(float(row["Cumulative_Return_%"]) - expected_returns[name]) < 0.02, (
        "Result differs from audited checkpoint: " + name
    )
print("PASS: 4,749 regime predictions, 4,747 next-open holding returns")
print("Date coverage:", final_comparison.index.min(), "to", final_comparison.index.max())
display(final_summary.round(3))


## 9. Final report visualisations

Portfolio equity and drawdown plots compare both pre-specified strategies with/without HMM filtering. Also plot the **number of BIC-selected HMM states and retained features** without interpreting raw HMM state IDs as consistent across adjacent windows.

In [ ]:
import matplotlib.pyplot as plt

labels = {
    "SMA_Unfiltered":"SMA unfiltered",
    "SMA_Filtered":"SMA HMM filtered",
    "Breakout_Unfiltered":"Breakout unfiltered",
    "Breakout_Filtered":"Breakout HMM filtered"
}

plt.figure(figsize=(12,6))
for name, label in labels.items():
    plt.plot(final_comparison.index, final_comparison[f"{name}_Equity"], label=label)
plt.axhline(1.0, color="gray", linestyle="--", linewidth=1)
plt.title("Methodology-aligned HMM: equity curves (2006–2024)")
plt.xlabel("Execution date"); plt.ylabel("Equity (initial capital = 1)")
plt.grid(alpha=0.25); plt.legend(); plt.tight_layout()
plt.savefig(OUTPUT_DIR / "submission_equity_curves.png", dpi=250, bbox_inches="tight")
plt.show()

plt.figure(figsize=(12,6))
for name, label in labels.items():
    equity = final_comparison[f"{name}_Equity"]
    rolling_peak = equity.cummax().clip(lower=1.0)
    plt.plot(final_comparison.index, (equity/rolling_peak-1)*100, label=label)
plt.title("Methodology-aligned HMM: maximum drawdown comparison (2006–2024)")
plt.xlabel("Execution date"); plt.ylabel("Drawdown (%)")
plt.grid(alpha=0.25); plt.legend(); plt.tight_layout()
plt.savefig(OUTPUT_DIR / "submission_drawdown_curves.png", dpi=250, bbox_inches="tight")
plt.show()

state_summary = corrected_predictions["Selected_States"].value_counts().sort_index()
feature_summary = corrected_predictions["Feature_Count"].value_counts().sort_index()
print("HMM state counts:"); display(state_summary.rename("Windows").to_frame())
print("Retained feature counts:"); display(feature_summary.rename("Windows").to_frame())


## 10. Cost and statistical return assessment

Gross strategy returns exclude simulated trading friction; net returns include 5 bps transaction cost plus 2 bps slippage **per position change**. Cost impact in percentage points is the difference between compounded gross and net cumulative returns; it is not the sum of per-event fees. PSR is calculated against a zero-Sharpe benchmark with a skew/kurtosis approximation; a numerical DSR **is not reported** because a defensible effective number of independent trials was not established (the two strategies and feature candidates were supervisor-specified, rather than profitability-selected).

In [ ]:
from scipy.stats import norm

locs = strategy_data.index.get_indexer(pd.DatetimeIndex(corrected_predictions["Prediction_Date"]))
assert (locs >= 0).all()
valid = locs + 2 < len(strategy_data)
preds = corrected_predictions.iloc[np.flatnonzero(valid)].reset_index(drop=True)
idx = locs[valid]
opens = strategy_data["Open"].to_numpy(dtype=float)
market_returns = opens[idx+2] / opens[idx+1] - 1
assert len(market_returns) == len(final_comparison)
np.testing.assert_allclose(final_comparison["Market_Return"], market_returns, atol=1e-12, rtol=0)

signal_columns = {
    "SMA_Unfiltered": "SMA_Signal",
    "SMA_Filtered": "SMA_Filtered_Signal",
    "Breakout_Unfiltered": "Breakout_Signal",
    "Breakout_Filtered": "Breakout_Filtered_Signal"
}
cost_records = []
statistics = []
for name, col in signal_columns.items():
    positions = preds[col].to_numpy(dtype=int)
    changes = np.abs(positions - np.r_[0, positions[:-1]])
    gross_daily = positions * market_returns
    net_daily = gross_daily - changes*TOTAL_COST
    np.testing.assert_allclose(final_comparison[f"{name}_Return"], net_daily, atol=1e-12, rtol=0)
    gross = (np.prod(1+gross_daily)-1)*100
    net = (np.prod(1+net_daily)-1)*100
    cost_records.append({
        "Strategy":name,
        "Gross_Return_%":gross,
        "Net_Return_%":net,
        "Gross_minus_Net_pp":gross-net,
        "Position_Changes":int(changes.sum()),
        "Charged_Cost_Sum_%":changes.sum()*TOTAL_COST*100
    })
    r = pd.Series(net_daily)
    sr = r.mean()/r.std(ddof=1)
    gamma3 = r.skew()
    gamma4 = r.kurt()+3
    denom = 1-gamma3*sr+((gamma4-1)/4)*sr**2
    psr = norm.cdf(sr*np.sqrt(len(r)-1)/np.sqrt(denom)) if denom>0 else np.nan
    statistics.append({
        "Strategy":name,
        "Annualised_Sharpe":sr*np.sqrt(252),
        "Skewness":gamma3,
        "Excess_Kurtosis":r.kurt(),
        "Lag1_Autocorrelation":r.autocorr(lag=1),
        "PSR_Above_Zero":psr,
        "DSR":np.nan
    })

cost_table = pd.DataFrame(cost_records)
sharpe_table = pd.DataFrame(statistics)
print("Gross / net cost reconciliation:"); display(cost_table.round(3))
print("Statistical assessment:"); display(sharpe_table.round(5))


## 11. Save final outputs and reproducibility manifest

Everything is written into `final_report_outputs/` so the original exploratory CSVs and notebook stay untouched. Historical Stooq price cross-check, NYSE calendar verification and event-specific outlier evidence are retained as **separate** supporting files, not inputs for model fitting.

In [ ]:
import sys
import json
import platform
import importlib.metadata as metadata

final_summary.to_csv(OUTPUT_DIR / "submission_performance.csv", index=False)
final_comparison.to_csv(OUTPUT_DIR / "submission_daily_backtest.csv")
cost_table.to_csv(OUTPUT_DIR / "submission_cost_audit.csv", index=False)
sharpe_table.to_csv(OUTPUT_DIR / "submission_sharpe_assessment.csv", index=False)
corrected_predictions.to_csv(OUTPUT_DIR / "submission_predictions_copy.csv", index=False)

manifest = {
    "project": "Gaussian HMM S&P500 regime filtering - Deliverable 4",
    "source_data": "hmm_dataset.csv",
    "prediction_checkpoint": str(PREDICTION_FILE),
    "final_predictions": len(corrected_predictions),
    "final_backtest_holding_periods": len(final_comparison),
    "first_prediction": str(corrected_predictions["Prediction_Date"].min().date()),
    "last_prediction": str(corrected_predictions["Prediction_Date"].max().date()),
    "first_execution": str(final_comparison.index.min().date()),
    "last_execution": str(final_comparison.index.max().date()),
    "candidate_features": CANDIDATE_HMM_FEATURES,
    "training_observations": INITIAL_WINDOW,
    "regime_state_candidates": list(HMM_STATE_COUNTS),
    "feature_correlation_threshold": CORRELATION_LIMIT,
    "feature_priority": FEATURE_PRIORITY,
    "HMM_model_selection": "BIC, 20 initialisations per state count",
    "historical_favourable_regime_rule": "Positive mean gross open-to-open return in training window for each strategy",
    "transaction_cost_per_change": TRANSACTION_COST,
    "slippage_per_change": SLIPPAGE,
    "DSR": "Not reported: effective number of independent performance-selection trials not established",
    "training_pilot_period_excluded": "2000-2005 excluded from final train/test rows",
    "python_version": sys.version.split()[0],
    "platform": platform.platform(),
    "packages": {p:metadata.version(p) for p in
                 ("numpy", "pandas", "scikit-learn", "hmmlearn", "scipy", "matplotlib")},
    "verified_expected_net_return_percent": expected_returns,
    "evidence_files": [
        "sp500_secondary_verification_summary.csv",
        "outlier_event_review_final_2026-10-21.csv",
        "final_exchange_calendar_audit.csv"
    ]
}
with open(OUTPUT_DIR / "submission_experiment_manifest.json", "w", encoding="utf-8") as fp:
    json.dump(manifest, fp, indent=2)

print("SUBMISSION NOTEBOOK: validation and export completed")
print("Report tables and figures written to", OUTPUT_DIR.resolve())


## 12. Main findings and methodological qualifications

| Strategy | Net return | Sharpe | Max drawdown |
|---|---:|---:|---:|
| SMA unfiltered | **128.992%** | **0.446** | **−29.432%** |
| SMA HMM filtered | **21.577%** | **0.166** | **−18.950%** |
| Breakout unfiltered | **98.954%** | **0.396** | **−40.990%** |
| Breakout HMM filtered | **40.293%** | **0.261** | **−29.462%** |

Filtering improved **maximum drawdown** but **did not improve Sharpe-based risk-adjusted returns**. The filtered strategies also increased trading frequency and total transaction-cost exposure. This is a valid empirical result, not a reason to tune away unfavourable findings.

**Qualifications:** (a) D3 prose inconsistently says 30 training days but uses the inclusive 31-observation example, which this notebook follows; (b) observed high HMM posterior probabilities do not independently establish regime validity; (c) historical market index returns are a simulation, not directly tradable index transactions; (d) the PSR reported above is not a DSR, and serial-dependence corrections are not applied; (e) the archived Stooq dataset is third-party provenance and the research-data licensing terms should be reviewed prior to external redistribution; (f) the submitted candidate feature set uses a diagonal covariance HMM, and some windows contain only one retained feature.